Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [ ]:
# Instalar ou atualizar bibliotecas necessárias: datasets do Hugging Face, transformers, torch e google.
#!pip install -q transformers accelerate bitsandbytes
!pip install -q -U google-genai

# Importar biblioteca pandas, função genai da biblioteca google e função userdata da biblioteca google.colab.
import pandas as pd
#import torch
from google import genai
from google.colab import userdata
#from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.9/750.9 kB 39.9 MB/s eta 0:00:00


Dataset com as 210 questões disponíveis no github.

In [ ]:
# URL das questões.
url_questions = "https://raw.githubusercontent.com/maritaca-ai/oab-bench/main/data/oab_bench/question.jsonl"

# Carregar um Dataframe com as perguntas a partir do arquivo .jsonl.
df_questions = pd.read_json(url_questions, lines=True)

# Inserir uma coluna para enumerar as questões.
df_questions.insert(0, 'question', range(1, len(df_questions)+1))

# Exibir o total de linhas.
print(f"Total de perguntas: {len(df_questions)}")

# Exibir as 5 primeiras linhas para ter noção do conteúdo.
display(df_questions.head())


Total de perguntas: 210


,question,question_id,category,statement,turns,values,system
0,1,39_direito_administrativo_peca_profissional,39_direito_administrativo,PEÇA PRÁTICO-PROFISSIONAL\n\nEm fevereiro de 2...,[],[5.0],Você é um bacharel em direito que está realiza...
1,2,39_direito_administrativo_questao_1,39_direito_administrativo,QUESTÃO\n\nA União fez publicar um edital de l...,[O edital em questão deveria contemplar a matr...,"[0.65, 0.6000000000000001]",Você é um bacharel em direito que está realiza...
2,3,39_direito_administrativo_questao_2,39_direito_administrativo,QUESTÃO\n\nA sociedade empresária Alfa foi con...,[A contratada tem direito à extinção do contra...,"[0.65, 0.6000000000000001]",Você é um bacharel em direito que está realiza...
3,4,39_direito_administrativo_questao_3,39_direito_administrativo,QUESTÃO\n\nJaqueline é servidora pública ocupa...,"[Jaqueline, como agente público responsável pe...","[0.65, 0.6000000000000001]",Você é um bacharel em direito que está realiza...
4,5,39_direito_administrativo_questao_4,39_direito_administrativo,QUESTÃO\n\nApós realizar pedido administrativo...,[Qual o prazo para a interposição do recurso a...,"[0.6000000000000001, 0.65]",Você é um bacharel em direito que está realiza...


Dataset com as 210 respostas de referências disponíveis no github.

In [ ]:
# URL das respostas de referência.
url_guidelines = "https://raw.githubusercontent.com/maritaca-ai/oab-bench/refs/heads/main/data/oab_bench/reference_answer/guidelines.jsonl"

# Carregar um Dataframe com as respostas a partir do arquivo .jsonl.
df_guidelines = pd.read_json(url_guidelines, lines=True)

# Exibir o total de linhas.
print(f"Total de respostas: {len(df_guidelines)}")

# Exibir as 5 primeiras linhas.
display(df_guidelines.head())

Total de respostas: 210


,question_id,answer_id,model_id,choices,tstamp
0,39_direito_administrativo_peca_profissional,2703a8c66fd84114b60e7ab713e4a9cc,guidelines,"[{'index': 0, 'turns': ['A peça a ser apresent...",1.686287e+09
1,39_direito_administrativo_questao_1,1ba691a0a6af496eb6b9a2b83b1ad689,guidelines,"[{'index': 0, 'turns': ['Sim. Nos contratos de...",NaN
2,39_direito_administrativo_questao_2,7524549417e94e03953b53d4e4964ad3,guidelines,"[{'index': 0, 'turns': ['Sim, a sociedade empr...",NaN
3,39_direito_administrativo_questao_3,8c128077f7794cc4b47e0cb659aedc01,guidelines,"[{'index': 0, 'turns': ['Sim, Jaqueline, como ...",NaN
4,39_direito_administrativo_questao_4,d61e1b8372534c19bf11f04614ffb3fa,guidelines,"[{'index': 0, 'turns': ['Considerando que não ...",NaN


Dataframes de questões e de referências do intervalo de um integrante da equipe.

In [ ]:
# Povoar subconjutos tanto das perguntas, quanto das respostas.
# Subconjunto das questões de 31 a 40.
# O índice do Dataframe começa do 0 (zero), portanto, a linha 31 para meu exercício é a 30 no pandas.
#   analogamente a 40 p é a 39 no pandas.
# iloc selciona um intervalo fechado à esquerda e aberto à direita, logo pegar o intervalo de 31 a 40 para mim
#   é no iloc de 30 (inclusive) até 40 (exclusive).

# Meu sub-grupo de questões e respostas.
df_my_questions = df_questions.iloc[30:40]
df_my_guidelines = df_guidelines.iloc[30:40]

Todas as categorias envolvendo as questões de um integrante.

In [ ]:
# Listar as diferentes categorias de perguntas do Dataframe de perguntas.
print(df_my_questions['category'].unique())

['39_direito_tributario' '40_direito_administrativo']


Diferentes perfis profissionais.

In [ ]:
# Listar os diferentes profissionais.
display(df_my_questions['system'].unique())

array(['Você é um bacharel em direito que está realizando a segunda fase da prova da Ordem dos Advogados do Brasil (OAB), organizada pela FGV. Sua tarefa é responder às questões discursivas e elaborar uma peça processual, demonstrando seu conhecimento jurídico, capacidade de raciocínio e habilidade de aplicar a legislação e jurisprudência pertinentes ao caso apresentado.\n\n# ATENÇÃO\n\nNa elaboração dos textos da peça prático-profissional e das respostas às questões discursivas, você deverá incluir todos os dados que se façam necessários, sem, contudo, produzir qualquer identificação ou informações além daquelas fornecidas e permitidas nos enunciados contidos no caderno de prova. A omissão de dados que forem legalmente exigidos ou necessários para a correta solução do problema proposto acarretará em descontos na pontuação atribuída a você nesta fase. Você deve estar atento para não gerar nenhum dado diferente que dê origem a uma marca identificadora.\n\nA detecção de qualquer marca id

Junção dos Dataframes de questoes e respostas de referências colocadas lado a lado em um novo e único Dataframe.

In [ ]:
# Junção dos DataFrame df_my_question e df_my_guidelines em um único DataFrame usando o question_id de ambos como atributo da interseção.
# Neste Dataframe dá para ver a pergunta e a resposta da linha guia.
df_question_and_guidelines = pd.merge(df_my_questions, df_my_guidelines, on='question_id', how='inner')

# Exibir o total de linhas.
print(f"Total de linha após a junção: {len(df_question_and_guidelines)}")

# Exibir o Dataframe das minhas questões.
display(df_question_and_guidelines)

Total de linha após a junção: 10


,question,question_id,category,statement,turns,values,system,answer_id,model_id,choices,tstamp
0,31,39_direito_tributario_peca_profissional,39_direito_tributario,"PEÇA PRÁTICO-PROFISSIONAL\n\nPedro de Camões, ...",[],[5.0],Você é um bacharel em direito que está realiza...,d2731b2abccb48a1a6e0137fd701faf2,guidelines,"[{'index': 0, 'turns': ['A medida judicial cab...",1.686287e+09
1,32,39_direito_tributario_questao_1,39_direito_tributario,QUESTÃO\n\nA sociedade empresária Metalúrgica ...,"[A partir de quando se deve contar, no caso co...","[0.6000000000000001, 0.65]",Você é um bacharel em direito que está realiza...,e4d432017edb48c3b3f2c35a278821ca,guidelines,"[{'index': 0, 'turns': ['O prazo para oferta d...",NaN
2,33,39_direito_tributario_questao_2,39_direito_tributario,QUESTÃO\n\nA sociedade empresária Tudo Verde L...,[A qual dos municípios o ISS de jardinagem é d...,"[0.65, 0.6000000000000001]",Você é um bacharel em direito que está realiza...,8cdbaa34517d4191b4ee99cda90cf09c,guidelines,"[{'index': 0, 'turns': ['O ISS de jardinagem é...",NaN
3,34,39_direito_tributario_questao_3,39_direito_tributario,QUESTÃO\n\nA Administração Tributária Federal ...,[É válida a exigência de depósito do montante ...,"[0.65, 0.6000000000000001]",Você é um bacharel em direito que está realiza...,72f15c31ec404271b2bd21eb0c9a7f2f,guidelines,"[{'index': 0, 'turns': ['Não, pois é inconstit...",NaN
4,35,39_direito_tributario_questao_4,39_direito_tributario,QUESTÃO\n\nA sociedade empresária Faz Tudo Ltd...,[Está correto o argumento da necessidade de no...,"[0.65, 0.6000000000000001]",Você é um bacharel em direito que está realiza...,0da4e43c949d48a49125bc7e98a4a82f,guidelines,"[{'index': 0, 'turns': ['Não está correto, por...",NaN
5,36,40_direito_administrativo_peca_profissional,40_direito_administrativo,PEÇA PRÁTICO-PROFISSIONAL\n\nO Ministério Públ...,[],[5.0],Você é um bacharel em direito que está realiza...,3e5ccfa8be5a4a298c51bce8a43a8b0e,guidelines,"[{'index': 0, 'turns': ['O(a) examinando(a) de...",1.686287e+09
6,37,40_direito_administrativo_questao_1,40_direito_administrativo,QUESTÃO\n\nA sociedade empresária Sagaz S.A. e...,[Há necessidade de demonstração do elemento su...,"[0.65, 0.6000000000000001]",Você é um bacharel em direito que está realiza...,0f4fed154f1d417e8b3110ce0cfc528d,guidelines,"[{'index': 0, 'turns': ['Não. A responsabiliza...",NaN
7,38,40_direito_administrativo_questao_2,40_direito_administrativo,QUESTÃO\n\nDeterminada informação de interesse...,[É lícita a cobrança efetuada pelo órgão respo...,"[0.65, 0.6000000000000001]",Você é um bacharel em direito que está realiza...,5132c7e50543433cb94804c1377a6245,guidelines,"[{'index': 0, 'turns': ['Não. A submissão e o ...",NaN
8,39,40_direito_administrativo_questao_3,40_direito_administrativo,QUESTÃO\n\nCerta Secretaria do Estado Alfa fez...,[É possível a utilização do sistema de registr...,"[0.65, 0.6000000000000001]",Você é um bacharel em direito que está realiza...,09f14657abe143bf95ef9cbf9c7e6efb,guidelines,"[{'index': 0, 'turns': ['Sim. A Administração ...",NaN
9,40,40_direito_administrativo_questao_4,40_direito_administrativo,"QUESTÃO\n\nRecentemente, Iná foi aprovada em c...",[A aprovação de Iná no mencionado concurso imp...,"[0.65, 0.6000000000000001]",Você é um bacharel em direito que está realiza...,f2b8ee4a551349eb99e810e0e8df1a70,guidelines,"[{'index': 0, 'turns': ['Não. Iná foi aprovada...",NaN


Google Gemini em nuvém, configuração, definição do modelo e execução da consulta.

In [ ]:
# Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
# Tal chave previamente criada no Google AI Studio.
gemini_api_key = userdata.get('Gemma3')

# Inicializar o cliente da API.
client_ai = genai.Client(api_key=gemini_api_key)

# Modelo para rodar com CPU, limitado gratuitamente à 20 requisições.
model_id = 'gemini-flash-latest'

# Criar uma lista vazia, para guardar as respostas, por questão de performance.
gemini_responses = []

# Repetição que percorre todo Dataframe.
for index, row in df_question_and_guidelines.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    papel = row['system']
    categoria = row['category']
    contexto = row['statement']
    instrucao = row['turns']

    # Preparação do prompt em Markdown.
    prompt_usuario = f"""
    ### PAPEL:
    {papel}

    ### ÁREA:
    {categoria}

    ### CONTEXTO:
    {contexto}

    ### INSTRUÇÃO:
    {instrucao}
    """

    # Chamada simples para a API da Google em nuvem.
    response = client_ai.models.generate_content(
        model=model_id,
        contents=prompt_usuario,
        config={
            "temperature": 0.1,  # Mantém a precisão.
            "max_output_tokens": 1024
        }
    )

    # Armazenar o resultado em uma lista.
    gemini_responses.append({"question": questao, "response": response})
    print(f"Questão {questao} processada com sucesso.")

# Fechar conexão com a IA
client_ai.close()

# Para melhor visualização, converter as respostas em um Dataframe.
df_gemini_response = pd.DataFrame(gemini_responses)

Questão 31 processada com sucesso.
Questão 32 processada com sucesso.
Questão 33 processada com sucesso.
Questão 34 processada com sucesso.
Questão 35 processada com sucesso.
Questão 36 processada com sucesso.
Questão 37 processada com sucesso.
Questão 38 processada com sucesso.
Questão 39 processada com sucesso.
Questão 40 processada com sucesso.


Realização de perguntas, uma por uma, por API. Em construção. Não execute.

In [ ]:
# Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
# Tal chave previamente criada no Google AI Studio.
gemini_api_key = userdata.get('Gemma3')

# Inicializar o cliente da API.
client_ai = genai.Client(api_key=gemini_api_key)

# Modelo para rodar com CPU, limitado gratuitamente à 20 requisições.
#model = 'gemini-flash-latest'

# Modelo para rodar com GPUs:T4. Limite de requisições gratuitas -> 250.
model_id = 'google/gemma-3-27b-it'
#model_id = "google/gemma-3-4b-it" # Exemplo com a versão 4B Instruct

# O modelo selecionado tem 27 bilhões de parâmetros e não é possível executar na GPU T4 do Google Colab,
# por isso o uso da tokenização.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Configuração da quantização para caber na GPU T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Definição do Modelo 27B
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

# Criar uma lista vazia, para guardar as respostas, por questão de performance.
gemini_responses = []

# Repetição para percorrer todo Dataframe.
for index, row in df_question_and_guidelines.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    papel = row['system']
    categoria = row['category']
    contexto = row['statement']
    instrucao = row['turns']

    # Preparação do prompt em Markdown
    prompt_usuario = f"""
    ### PAPEL:
    {papel}

    ### ÁREA DE ATUAÇÃO:
    {categoria}

    ### CONTEXTO:
    {contexto}

    ### INSTRUÇÃO:
    {instrucao}
    """

    # Preparação da template para o chat.
    messages = [{"role": "user", "content": prompt_usuario}]
    prompt_formatado = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Geração da resposta.
    inputs = tokenizer(prompt_formatado, return_tensors="pt").to("cuda")

    with torch.no_grad(): # Economiza memória ao não calcular gradientes
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1, # Mantém a precisão técnica
            do_sample=False  # Garante respostas mais determinísticas para questões técnicas.
        )


    # Decodificação e Limpeza
    full_respense = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extrai apenas a parte que vem depois da tag do modelo
    clear_response = full_respense.split("model\n")[-1].strip()

    # Armazenar o resultado em uma lista.
    gemini_responses.append({"question": questao, "response": clear_response})
    print(f"Questão {questao} processada com sucesso.")

# Fechar conexão com a IA
client_ai.close()

# Para melhor visualização, converter as respostas em um Dataframe.
df_gemini_response = pd.DataFrame(gemini_responses)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-3-27b-it.
401 Client Error. (Request ID: Root=1-69c70087-0256ddc576fac41a2ab7b8ca;70c0a1a9-0a22-4e23-bbf1-ad1bc09af715)

Cannot access gated repo for url https://huggingface.co/google/gemma-3-27b-it/resolve/main/config.json.
Access to model google/gemma-3-27b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
# Exibir as 5 primeiras linhas de respostas.
df_gemini_response.head()

,question,response
0,31,sdk_http_response=HttpResponse(\n headers=<di...
1,32,sdk_http_response=HttpResponse(\n headers=<di...
2,33,sdk_http_response=HttpResponse(\n headers=<di...
3,34,sdk_http_response=HttpResponse(\n headers=<di...
4,35,sdk_http_response=HttpResponse(\n headers=<di...


In [ ]:
# Objetos mais relevantes até aqui.

# DataFrame com perguntas e respostas da linha guia.
df_question_and_guidelines.info()

# Dataframe com as respostas do gemini, com o campo question para relacionar com o dataframe das perguntas.
df_gemini_response.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   question     10 non-null     int64  
 1   question_id  10 non-null     object 
 2   category     10 non-null     object 
 3   statement    10 non-null     object 
 4   turns        10 non-null     object 
 5   values       10 non-null     object 
 6   system       10 non-null     object 
 7   answer_id    10 non-null     object 
 8   model_id     10 non-null     object 
 9   choices      10 non-null     object 
 10  tstamp       2 non-null      float64
dtypes: float64(1), int64(1), object(9)
memory usage: 1012.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  10 non-null     int64 
 1   response  10 non-null     object
dtypes: int64(1), object(1)
mem